# Origin Classification with Static Word Embeddings

Combinations: Word2Vec, GloVe, fastText with Logistic Regression and Linear SVM.

In [11]:
import re
import numpy as np
import pandas as pd

from gensim.models import Word2Vec, FastText
import gensim.downloader as api

from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.model_selection import train_test_split

In [12]:
DATA_PATH = 'Data/final_coffee_reviews.csv'
RANDOM_STATE = 42
MIN_SAMPLES_PER_CLASS = 50
VECTOR_SIZE = 100

df = pd.read_csv(DATA_PATH)
print('Shape:', df.shape)
df.head(2)

Shape: (7585, 25)


,URL,all_text,Rating,Roaster,Coffee Name,Roaster Location,Coffee Origin,Roast Level,Agtron,Est. Price,...,Flavor,Aftertaste,With Milk,Blind Assessment,Notes,Who Should Drink It,Bottom Line,Combined_Acidity,Country_all,Country
0,https://www.coffeereview.com/review/100-arabic...,87\nCaribeans\n100% Arabica Coffee from Puerto...,87.0,Caribeans,100% Arabica Coffee from Puerto Rico,"San Juan, Puerto Rico","Utuado, central Puerto Rico",Medium-Light,54/69,$17.00/8 ounces,...,8.0,7.0,NaN,bittersweet but balanced; chocolaty. dark choc...,Produced on a single farm in the central mount...,NaN,Satisfying chocolate and nut notes nearly carr...,7.0,Puerto Rico; China,Puerto Rico
1,https://www.coffeereview.com/review/100-arabic...,88\nWaka Coffee\n100% Arabica Freeze-Dried Col...,88.0,Waka Coffee,100% Arabica Freeze-Dried Colombian (Instant C...,"Los Angeles, California",Colombia,NaN,0/0,$10.99/8 single-serve packets,...,8.0,8.0,NaN,evaluated at proportions of 5 grams of instant...,The green coffee for this product was produced...,NaN,A appealing 100% Colombia coffee in instant fo...,7.0,Colombia,Colombia


In [13]:
def normalize_text(text: str) -> str:
    if not isinstance(text, str):
        return ''
    text = text.lower().strip()
    text = re.sub(r'http\S+|www\.\S+', ' ', text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text


def tokenize_text(text: str):
    return [w for w in text.split() if len(w) > 1]


SPECIAL_COUNTRIES = ['papua new guinea', 'costa rica', 'el salvador']

def extract_country(origin):
    if pd.isna(origin):
        return None
    s = str(origin).strip().lower().rstrip('.')
    if not s:
        return None
    if re.search(r'not disclosed|blend|various|multiple', s):
        return None

    for c in SPECIAL_COUNTRIES:
        if re.search(rf'\b{re.escape(c)}\b', s):
            return c.title()

    parts = s.split()
    if not parts:
        return None

    country = parts[-1]
    if country in {'hawaii', "hawai'i"}:
        return 'United States'
    return country.title()

In [14]:
# Build text and labels
df['text_main'] = df['Blind Assessment'].fillna('').map(normalize_text)

if 'Country' in df.columns:
    df['origin_country'] = df['Country'].map(extract_country)
else:
    df['origin_country'] = df['Coffee Origin'].map(extract_country)

work = df[(df['text_main'].str.len() >= 30) & (df['origin_country'].notna())].copy()
counts = work['origin_country'].value_counts()
work = work[work['origin_country'].isin(counts[counts >= MIN_SAMPLES_PER_CLASS].index)].copy()

work['tokens'] = work['text_main'].map(tokenize_text)

print('Rows after filtering:', len(work))
print('Num classes:', work['origin_country'].nunique())
work['origin_country'].value_counts().head(15)

Rows after filtering: 7317
Num classes: 22


origin_country
Ethiopia       1996
Colombia        988
Kenya           699
Guatemala       561
Indonesia       452
Costa Rica      373
Panama          354
States          302
El Salvador     240
Brazil          200
Rwanda          176
Peru            130
Nicaragua       125
Honduras        123
Mexico          101
Name: count, dtype: int64

In [15]:
X_tokens = work['tokens']
y = work['origin_country']

X_train_tok, X_temp_tok, y_train, y_temp = train_test_split(
    X_tokens, y, test_size=0.30, random_state=RANDOM_STATE, stratify=y
)
X_val_tok, X_test_tok, y_val, y_test = train_test_split(
    X_temp_tok, y_temp, test_size=0.50, random_state=RANDOM_STATE, stratify=y_temp
)

print('Train:', len(X_train_tok), 'Val:', len(X_val_tok), 'Test:', len(X_test_tok))

Train: 5121 Val: 1098 Test: 1098


In [16]:
def docs_to_avg_vectors(token_docs, kv, vector_size=100):
    X = np.zeros((len(token_docs), vector_size), dtype=np.float32)
    for i, tokens in enumerate(token_docs):
        vecs = [kv[w] for w in tokens if w in kv.key_to_index]
        if vecs:
            X[i] = np.mean(vecs, axis=0)
    return X

In [17]:
# Train/load static embeddings
w2v_model = Word2Vec(
    sentences=X_train_tok.tolist(),
    vector_size=VECTOR_SIZE,
    window=5,
    min_count=2,
    workers=4,
    sg=1,
    seed=RANDOM_STATE
)

ft_model = FastText(
    sentences=X_train_tok.tolist(),
    vector_size=VECTOR_SIZE,
    window=5,
    min_count=2,
    workers=4,
    sg=1,
    seed=RANDOM_STATE
)

glove_kv = None
try:
    glove_kv = api.load('glove-wiki-gigaword-100')
    print('Loaded GloVe: glove-wiki-gigaword-100')
except Exception as e:
    print('Could not load GloVe. Word2Vec and fastText will still run.')
    print('Reason:', e)

Loaded GloVe: glove-wiki-gigaword-100


In [18]:
embedding_sources = {
    'Word2Vec': w2v_model.wv,
    'fastText': ft_model.wv,
}
if glove_kv is not None:
    embedding_sources['GloVe'] = glove_kv

vector_cache = {}
for emb_name, kv in embedding_sources.items():
    vector_cache[(emb_name, 'train')] = docs_to_avg_vectors(X_train_tok.tolist(), kv, VECTOR_SIZE)
    vector_cache[(emb_name, 'val')] = docs_to_avg_vectors(X_val_tok.tolist(), kv, VECTOR_SIZE)
    vector_cache[(emb_name, 'test')] = docs_to_avg_vectors(X_test_tok.tolist(), kv, VECTOR_SIZE)

print('Prepared vector matrices for embeddings:', list(embedding_sources.keys()))

Prepared vector matrices for embeddings: ['Word2Vec', 'fastText', 'GloVe']


In [ ]:
def evaluate_predictions(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)
    return acc, p, r, f1

classifiers = {
    'LogisticRegression': LogisticRegression(max_iter=3000, class_weight='balanced', solver='saga', n_jobs=-1, random_state=RANDOM_STATE),
    'LinearSVM': LinearSVC(C=1.0, class_weight='balanced', dual='auto', max_iter=10000, random_state=RANDOM_STATE),
}

results = []
for emb_name in embedding_sources.keys():
    Xtr = vector_cache[(emb_name, 'train')]
    Xv = vector_cache[(emb_name, 'val')]
    Xte = vector_cache[(emb_name, 'test')]

    for clf_name, clf in classifiers.items():
        clf.fit(Xtr, y_train)
        y_val_pred = clf.predict(Xv)
        val_acc, val_p, val_r, val_f1 = evaluate_predictions(y_val, y_val_pred)
        y_test_pred = clf.predict(Xte)
        test_acc, test_p, test_r, test_f1 = evaluate_predictions(y_test, y_test_pred)

        results.append({
            'embedding': emb_name,
            'classifier': clf_name,
            'val_accuracy': val_acc,
            'val_precision_macro': val_p,
            'val_recall_macro': val_r,
            'val_f1_macro': val_f1,
            'test_accuracy': test_acc,
            'test_precision_macro': test_p,
            'test_recall_macro': test_r,
            'test_f1_macro': test_f1,
        })

results_df = pd.DataFrame(results).sort_values('test_f1_macro', ascending=False).reset_index(drop=True)
results_df

,embedding,classifier,val_accuracy,val_precision_macro,val_recall_macro,val_f1_macro,test_accuracy,test_precision_macro,test_recall_macro,test_f1_macro
0,Word2Vec,LinearSVM,0.168488,0.125635,0.134992,0.091127,0.193989,0.144708,0.167579,0.110072
1,GloVe,LogisticRegression,0.113843,0.102633,0.130017,0.083828,0.132058,0.148708,0.139190,0.102313
2,Word2Vec,LogisticRegression,0.132058,0.126726,0.150611,0.088071,0.151184,0.148212,0.151268,0.100486
3,fastText,LinearSVM,0.171220,0.091433,0.137473,0.086243,0.191257,0.127044,0.143165,0.099358
4,fastText,LogisticRegression,0.122951,0.100405,0.150828,0.083199,0.142987,0.134794,0.131563,0.088206
5,GloVe,LinearSVM,0.146630,0.098486,0.132567,0.090335,0.130237,0.118649,0.112545,0.083081


: 